# ProximityPrep — exploration

Étape 3 : distances plage et comptages commerces de proximité.

**Contrat (comme MeteoPrep)** :
- `hotel_code` = code Accor RodPrep (`H2075`, …) — jamais un nom ni un slug
- `hotel_lat` / `hotel_lon` fournis par RodPrep
- géocodage par nom uniquement si coords absentes

Objectif : visualiser entrées / sorties et remplir `../Output/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd().resolve()
while ROOT.name != "ProximityPrep" and ROOT.parent != ROOT:
    ROOT = ROOT.parent
PREPARE = ROOT.parent
PROJECT = PREPARE.parent

sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(ROOT / "Src"))

INPUT_DIR = ROOT / "Input"
OUTPUT_DIR = ROOT / "Output"
ROD_OUTPUT = PREPARE / "RodPrep" / "Output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

## 1. Entrée — identité depuis RodPrep

`fill_input_from_rod` ne garde que les colonnes d'identité et drop les lignes sans `hotel_code` Accor.

In [ ]:
from proximity_prep.prep import HOTEL_IDENTITY_COLS, ProximityPrep

prep = ProximityPrep(INPUT_DIR, OUTPUT_DIR)

if not (ROD_OUTPUT / "hotel_lookup.parquet").exists():
    raise FileNotFoundError("Exécuter d'abord RodPrep")

# Toujours resynchroniser depuis RodPrep (évite les anciens slugs en Input)
hotels_path = prep.fill_input_from_rod(ROD_OUTPUT)
print("Entrée synchronisée :", hotels_path)

hotels = prep.load_input()
print(f"Hôtels : {len(hotels)}")
print("Colonnes :", list(hotels.columns))
print("Codes Accor :", hotels["hotel_code"].tolist())
print("Coords manquantes :", hotels["hotel_lat"].isna().sum())
hotels

## 2. Ligne par hôtel (coords RodPrep prioritaires)

Inspection d'un hôtel : `geo_source` doit être `rod_coords` si lat/lon présents.

In [ ]:
sample = hotels.iloc[0]
row = prep._row_for_hotel(sample)
print("hotel_code :", row["hotel_code"])
print("geo_source :", row["geo_source"])
print("lat/lon    :", row["hotel_lat"], row["hotel_lon"])
print("plage_km   :", row.get("plage_distance_km"))
print("fb 100/500 :", row.get("commerce_fb_100m"), row.get("commerce_fb_500m"))
pd.Series(row)

## 3. Pipeline complet → Output/

In [ ]:
proximity = prep.run()
print(f"Lignes : {len(proximity)}")
print("geo_source :", proximity["geo_source"].value_counts().to_dict())
proximity

## 4. Contrôles qualité

- aucun slug / nom dans `hotel_code`
- jointure possible avec MeteoPrep / SalesPrep sur le même code Accor

In [ ]:
assert proximity["hotel_code"].notna().all()
assert not proximity["hotel_code"].astype(str).str.contains(r"^[a-z].*-", regex=True).any(), (
    "hotel_code ressemble à un slug registre"
)
assert (proximity["hotel_code"] != proximity["hotel_name"]).all()

meteo_codes = set()
meteo_path = PREPARE / "MeteoPrep" / "Output" / "meteo_monthly.parquet"
if meteo_path.exists():
    meteo_codes = set(pd.read_parquet(meteo_path)["hotel_code"].dropna().unique())
    overlap = set(proximity["hotel_code"]) & meteo_codes
    print(f"Intersection codes Accor Proximity ∩ Meteo : {sorted(overlap)}")
else:
    print("MeteoPrep Output absent — skip jointure check")

proximity[
    [
        "hotel_code",
        "hotel_name",
        "geo_source",
        "plage_distance_km",
        "commerce_fb_100m",
        "commerce_fb_500m",
        "commerce_non_fb_100m",
        "commerce_non_fb_500m",
    ]
]